# Assignment 1 — training pipeline

You're using a notebook for this stage. If you'd rather use a script,
write `train.py` instead and delete this file — `main.py` refuses to run
if it finds both `train.py` and `train.ipynb`, so exactly one of them must
exist.

Replace this cell and everything below it with your own code. When run via
`uv run python main.py train`, this notebook must, start to finish, with
no manual steps in between:

- Load your committed dataset.
- Apply the feature engineering, preprocessing, and train/val/test split
  consistent with the business framing you wrote in `REPORT.md`.
- Train the same model — linear regression for a regression target,
  logistic regression for a binary one — three ways, on the same split:
  scikit-learn, a from-scratch PyTorch loop (Session 5's manual approach:
  raw tensors, autograd, a manual gradient-step update), and the standard
  `torch.nn.Module` + `torch.optim` workflow.
- Evaluate all three against each other and against a naive baseline.
- Save whatever `app.py`/`app.ipynb` needs to build its dashboard without
  retraining anything.

How you structure the code beyond that — cell layout, function
boundaries, naming — is your call to make and be able to defend.

# Libraries

In [1]:
import json
import os

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, log_loss, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import joblib
from sklearn.preprocessing import StandardScaler

## EDA

In [2]:
df = pd.read_csv("data/Loan_default.csv")

print(df.shape) #did this primarily for the report as it asks the amount of rows and columns it has 
#also to see whether or not the dataset is large enough, but I will ask/asked you in class directly to make sure
#update: you told me the dataset is large enough when I asked you before class

# Missing values per column (only those with any), with percentage
na_counts = df.isna().sum()
print(na_counts.sum()) #shows that there are no na_values (which is great)

print(df.dtypes) 
print(df.head()) #from the first five rows we can see that a lot of the values are string values
#I can use one-hot encoding or change them to numeric values since,
#for that I would have to see how many different "types" of strings there are in each category
#for instance, bachelor's in education could be 1, high-school could be 0,

(255347, 18)
0
LoanID             object
Age                 int64
Income              int64
LoanAmount          int64
CreditScore         int64
MonthsEmployed      int64
NumCreditLines      int64
InterestRate      float64
LoanTerm            int64
DTIRatio          float64
Education          object
EmploymentType     object
MaritalStatus      object
HasMortgage        object
HasDependents      object
LoanPurpose        object
HasCoSigner        object
Default             int64
dtype: object
       LoanID  Age  Income  LoanAmount  CreditScore  MonthsEmployed  \
0  I38PQUQS96   56   85994       50587          520              80   
1  HPSK72WA7R   69   50432      124440          458              15   
2  C1OZ6DPJ8Y   46   84208      129188          451              26   
3  V2KKSFM3UN   32   31713       44799          743               0   
4  EY08JDHTZP   60   20437        9139          633               8   

   NumCreditLines  InterestRate  LoanTerm  DTIRatio    Education  \
0       

In [3]:
print(df["Education"].nunique()) #as previously stated, a lot of the columns are strings, so I will go one-by-one to see
#how many values there are in each column as a few of them are seeminly binary which means I'll just use one-hot encoding

print(df["Education"].value_counts(dropna=False)) #I can see there is a relatively even distribution of education values, perfect

4
Bachelor's     64366
High School    63903
Master's       63541
PhD            63537
Name: Education, dtype: int64


In [4]:
education_map_edu = {"High School": 0, "Bachelor's": 1, "Master's": 2, "PhD": 3,} #created a dictionary with the string options as planned
df["Education"] = df["Education"].map(education_map_edu) #applied changes
print(df["Education"].value_counts()) #added this to check whether or not the numbers correlated to the output above with the numerical encodings

1    64366
0    63903
2    63541
3    63537
Name: Education, dtype: int64


In [5]:
print(df["EmploymentType"].value_counts(dropna=False)) #not binary, so will apply same exact logic as I did previously

Part-time        64161
Unemployed       63824
Self-employed    63706
Full-time        63656
Name: EmploymentType, dtype: int64


In [6]:
education_map_employ = {"Unemployed": 0, "Self-employed": 1, "Part-time": 2, "Full-time": 3,} #same thing as before
df["EmploymentType"] = df["EmploymentType"].map(education_map_employ) #applied changes
print(df["EmploymentType"].value_counts()) #added this to check whether or not the numbers correlated to the output above with the numerical encodings

2    64161
0    63824
1    63706
3    63656
Name: EmploymentType, dtype: int64


In [7]:
print(df["MaritalStatus"].nunique()) #I initially assumed Marital Status would be binary, but I forgot divorced is an option. Will keep this to
#show my thought process
print(df["HasMortgage"].nunique()) #binary
print(df["HasDependents"].nunique()) #binary
print(df["HasCoSigner"].nunique()) #binary
#default column is alredy binary, which is perfect as that is my target variable
print(df["MaritalStatus"].value_counts(dropna=False))

3
2
2
2
Married     85302
Divorced    85033
Single      85012
Name: MaritalStatus, dtype: int64


In [8]:
education_map_marital = {"Single": 0, "Married": 1, "Divorced": 2} 
df["MaritalStatus"] = df["MaritalStatus"].map(education_map_marital) 
print(df["MaritalStatus"].value_counts()) #as with rest of columns, we can see an even distribution, should be good later down the line

1    85302
2    85033
0    85012
Name: MaritalStatus, dtype: int64


In [9]:
binary_cols = ["HasMortgage", "HasDependents", "HasCoSigner"]

for col in binary_cols:
    df[col] = (df[col].str.strip() == "Yes").astype(int)

print(df[binary_cols].dtypes)
print(df[binary_cols].head())

HasMortgage      int64
HasDependents    int64
HasCoSigner      int64
dtype: object
   HasMortgage  HasDependents  HasCoSigner
0            1              1            1
1            0              0            1
2            1              1            0
3            0              0            0
4            0              1            0


In [10]:
print(df.head()) #checked to make sure changes were applied as intended

       LoanID  Age  Income  LoanAmount  CreditScore  MonthsEmployed  \
0  I38PQUQS96   56   85994       50587          520              80   
1  HPSK72WA7R   69   50432      124440          458              15   
2  C1OZ6DPJ8Y   46   84208      129188          451              26   
3  V2KKSFM3UN   32   31713       44799          743               0   
4  EY08JDHTZP   60   20437        9139          633               8   

   NumCreditLines  InterestRate  LoanTerm  DTIRatio  Education  \
0               4         15.23        36      0.44          1   
1               1          4.81        60      0.68          2   
2               3         21.17        24      0.31          2   
3               3          7.07        24      0.23          0   
4               4          6.51        48      0.73          1   

   EmploymentType  MaritalStatus  HasMortgage  HasDependents LoanPurpose  \
0               3              2            1              1       Other   
1               3       

### Fixing my EDA

Most columns above I changed to one-hot encoding as that is easier for Logistic Regression models to handle

I kept the code above in order to showcase my throught process and show where I failed, and what I learned from this assignment
I hope this won't negatively impact my grade

In [11]:
df = pd.read_csv("data/Loan_default.csv")
df = df.drop(columns=["LoanID"])

order = ["High School", "Bachelor's", "Master's", "PhD"]
df["Education"] = pd.Categorical(df["Education"].str.strip(), categories=order, ordered=True).codes #ordinal

for col in ["HasMortgage", "HasDependents", "HasCoSigner"]:
    df[col] = (df[col].str.strip() == "Yes").astype(int) #one-hot

df = pd.get_dummies(df, columns=["EmploymentType", "MaritalStatus", "LoanPurpose"], drop_first=True, dtype=int) #one-hot

assert df.select_dtypes(include="object").shape[1] == 0, "string columns remain"
assert df.isna().sum().sum() == 0, "missing values remain"
print(df.shape)
print(df.head()) #checked to make sure changes were applied as intended

(255347, 23)
   Age  Income  LoanAmount  CreditScore  MonthsEmployed  NumCreditLines  \
0   56   85994       50587          520              80               4   
1   69   50432      124440          458              15               1   
2   46   84208      129188          451              26               3   
3   32   31713       44799          743               0               3   
4   60   20437        9139          633               8               4   

   InterestRate  LoanTerm  DTIRatio  Education  ...  Default  \
0         15.23        36      0.44          1  ...        0   
1          4.81        60      0.68          2  ...        0   
2         21.17        24      0.31          2  ...        1   
3          7.07        24      0.23          0  ...        0   
4          6.51        48      0.73          1  ...        0   

   EmploymentType_Part-time  EmploymentType_Self-employed  \
0                         0                             0   
1                         0  

### Subsampling
Subsampling: when I talked to you before one of our sessions, you told me it would be a good idea to subsample the data since it may be excessive, which is what I will be doing in the code cell below

In [12]:
df_small, _ = train_test_split(df, train_size=15000, stratify=df["Default"], random_state=42)
df_small = df_small.reset_index(drop=True)
df_small.to_csv("data/loan_sample.csv", index=False)

print(df_small.shape)
print(round(df_small["Default"].mean(),4)) 
print(round(df["Default"].mean(),4)) #the numbers after four decimal places diff, so I rounded it to show that my subsample is accurate in terms of average default rate

(15000, 23)
0.1161
0.1161


## Engineer features

These are row-wise calculations, so I did them before the split to avoid data leakage.

I discussed these added features in my report. They are all added because they make the model more robust and reliable from a financial point of view.

In [13]:
df = pd.read_csv("data/loan_sample.csv")

df["LoanToIncome"] = df["LoanAmount"] / df["Income"]

r = df["InterestRate"] / 100 / 12
df["MonthlyPayment"] = df["LoanAmount"] * r / (1 - (1 + r) ** (-df["LoanTerm"]))
df["PaymentToIncome"] = df["MonthlyPayment"] * 12 / df["Income"]

df["EmploymentShare"] = (df["MonthsEmployed"] / 12) / df["Age"]

df["LogIncome"] = np.log1p(df["Income"])
df["LogLoanAmount"] = np.log1p(df["LoanAmount"])

assert np.isfinite(df.drop(columns="Default")).all().all(), "inf or NaN created"
print(df[["LoanToIncome", "MonthlyPayment", "PaymentToIncome", "EmploymentShare"]].describe()) #sanity checks

       LoanToIncome  MonthlyPayment  PaymentToIncome  EmploymentShare
count  15000.000000    15000.000000     15000.000000     15000.000000
mean       2.191635     5660.783898         1.166695         0.130958
std        2.204154     4720.039220         1.496046         0.098196
min        0.033749       97.230960         0.009253         0.000000
25%        0.808035     2307.798443         0.335942         0.056054
50%        1.544700     4484.183333         0.686248         0.114035
75%        2.688409     7329.576666         1.395821         0.177778
max       16.231061    23508.030830        15.338433         0.550926


## Split 
Stratified sample split 70/15/15 = train/test/validation

In [14]:
X = df.drop(columns="Default")
y = df["Default"]

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.1765, stratify=y_trainval, random_state=42)

print(len(X_train), len(X_val), len(X_test))
print(y_train.mean(), y_val.mean(), y_test.mean()) 
#the three values are very similar but the y_value mean diff slightly

10499 2251 2250
0.11620154300409563 0.1159484673478454 0.116


## Scaling

Fitting on train only

Scaled the continuous columns and left the 0/1 columns alone:

In [15]:
binary_like = [c for c in X.columns if set(X[c].unique()) <= {0, 1}]
num_cols = [c for c in X.columns if c not in binary_like and c != "Education"]

scaler = StandardScaler().fit(X_train[num_cols])   # train only

def apply_scaler(X_part):
    X_part = X_part.copy()
    X_part[num_cols] = scaler.transform(X_part[num_cols])
    return X_part

X_train_s, X_val_s, X_test_s = map(apply_scaler, (X_train, X_val, X_test))

In [16]:
os.makedirs("models", exist_ok=True)
os.makedirs("data", exist_ok=True)

joblib.dump(scaler, "models/scaler.joblib")
json.dump({"feature_cols": list(X.columns), "num_cols": num_cols},
          open("models/preprocessing.json", "w"))

for name, Xp, yp in [("train", X_train_s, y_train), ("val", X_val_s, y_val), ("test", X_test_s, y_test)]:
    out = Xp.copy()
    out["Default"] = yp.values
    out.to_csv(f"data/{name}.csv", index=False)

# Three Logistic Regression Models

In [ ]:
SEED = 42
FN_COST = 10.0   #as explained in the report: approves a loan that defaults
FP_COST = 1.0    #per contra, declines a loan that would have been repaid
C_GRID = [0.001, 0.01, 0.1, 1.0, 10.0]   # egularization strength
LR_MANUAL = [0.01, 0.1, 1.0]
LR_STANDARD = [0.001, 0.01, 0.1]
MANUAL_EPOCHS = 1000
STANDARD_EPOCHS = 30
BATCH_SIZE = 256

np.random.seed(SEED)
torch.manual_seed(SEED)
os.makedirs("models", exist_ok=True)

#data loading
train = pd.read_csv("data/train.csv")
val = pd.read_csv("data/val.csv")
test = pd.read_csv("data/test.csv")
feature_cols = json.load(open("models/preprocessing.json"))["feature_cols"]

X_train, y_train = train[feature_cols].values.astype("float32"), train["Default"].values
X_val, y_val = val[feature_cols].values.astype("float32"), val["Default"].values
X_test, y_test = test[feature_cols].values.astype("float32"), test["Default"].values
n, d = X_train.shape
print(f"train={n}, val={len(X_val)}, test={len(X_test)}, features={d}")


#data evaluation
def cost_at(y_true, proba, threshold):
    pred = proba >= threshold
    fn = np.sum((y_true == 1) & ~pred)
    fp = np.sum((y_true == 0) & pred)
    return FN_COST * fn + FP_COST * fp


def best_threshold(y_true, proba):
    grid = np.linspace(0.01, 0.99, 99)
    costs = [cost_at(y_true, proba, t) for t in grid]
    return float(grid[int(np.argmin(costs))]) #threshold chosen that minimizes total cost on given set using validation set


def evaluate(y_true, proba, threshold): #evaluation metrics for Gradio dashboard
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    cost = FN_COST * fn + FP_COST * fp
    return {
        "roc_auc": float(roc_auc_score(y_true, proba)),
        "pr_auc": float(average_precision_score(y_true, proba)),
        "threshold": threshold,
        "cost": float(cost),
        "cost_per_loan": float(cost / len(y_true)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


base_rate = float(y_train.mean()) #naive baseline
base_val = np.full(len(y_val), base_rate)
base_test = np.full(len(y_test), base_rate)


best = None #scikit learn (first model)
for C in C_GRID:
    m = LogisticRegression(C=C, max_iter=2000)
    m.fit(X_train, y_train)
    ll = log_loss(y_val, m.predict_proba(X_val)[:, 1])
    print(f"sklearn C={C}: val log loss {ll:.4f}")
    if best is None or ll < best[0]:
        best = (ll, C, m)
_, best_C, sk_model = best
print(f"-> sklearn best C = {best_C}")
joblib.dump(sk_model, "models/sklearn_logreg.joblib")

sk_val = sk_model.predict_proba(X_val)[:, 1]
sk_test = sk_model.predict_proba(X_test)[:, 1]

# The PyTorch versions reuse the same regularization strength so the three
# methods fit the same objective. sklearn minimizes
#   C * sum(loss) + 0.5 * ||w||^2
# which is equivalent to  mean(loss) + 0.5 * lam * ||w||^2  with lam = 1 / (C * n).
lam = 1.0 / (best_C * n) 

Xt_train = torch.tensor(X_train)
yt_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
Xt_val = torch.tensor(X_val)
Xt_test = torch.tensor(X_test)


#manual PyTorch from session five notebook
yt_train_flat = torch.tensor(y_train, dtype=torch.float32)  #shape (n,) copies from the notebook


def forward_manual(X, w, b):
    return torch.sigmoid(X @ w + b) #copy-pasted from forward_clf from notebook


def bce_loss(y_pred, y_true):
    eps = 1e-7 #clamped to avoid log(0)
    y_pred = torch.clamp(y_pred, eps, 1 - eps) #binary cross-entropy
    return -(y_true * torch.log(y_pred) + (1 - y_true) * torch.log(1 - y_pred)).mean()


def train_manual(lr, epochs=MANUAL_EPOCHS): #manually updates epochs using gradient descent
    w = torch.zeros(d, requires_grad=True) #full-batch gradient descent
    b = torch.zeros(1, requires_grad=True)
    for _ in range(epochs):
        y_pred = forward_manual(Xt_train, w, b) #parameter updates
        loss = bce_loss(y_pred, yt_train_flat) + 0.5 * lam * (w ** 2).sum()
        loss.backward()
        with torch.no_grad():       #ensures that the update itself isn't tracked
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad.zero_()              #reset; otherwise, gradients accumulate
        b.grad.zero_()
    return w.detach(), b.detach()


def predict_manual(X, w, b):
    with torch.no_grad():
        return forward_manual(X, w, b).numpy()


best = None
for lr in LR_MANUAL:
    w, b = train_manual(lr)
    ll = log_loss(y_val, predict_manual(Xt_val, w, b))
    print(f"manual lr={lr}: val log loss {ll:.4f}")
    if best is None or ll < best[0]:
        best = (ll, lr, w, b)
_, best_lr_manual, w_man, b_man = best
print(f"-> manual best lr = {best_lr_manual}")
torch.save({"w": w_man, "b": b_man}, "models/manual_torch.pt")

man_val = predict_manual(Xt_val, w_man, b_man)
man_test = predict_manual(Xt_test, w_man, b_man)



def train_standard(lr, epochs=STANDARD_EPOCHS): #standard PyTorch with Adam optimizer
    torch.manual_seed(SEED)
    model = nn.Linear(d, 1)
    optimizer = torch.optim.Adam( #Adam optimizer
        [
            {"params": [model.weight], "weight_decay": lam},
            {"params": [model.bias], "weight_decay": 0.0},
        ],
        lr=lr,
    )
    loss_fn = nn.BCEWithLogitsLoss() #L2 regularization applied on weights only, not bias
    loader = DataLoader(TensorDataset(Xt_train, yt_train), batch_size=BATCH_SIZE, shuffle=True)
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
    return model


def predict_standard(model, X):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(X)).squeeze(1).numpy()


best = None
for lr in LR_STANDARD:
    model = train_standard(lr)
    ll = log_loss(y_val, predict_standard(model, Xt_val))
    print(f"standard lr={lr}: val log loss {ll:.4f}")
    if best is None or ll < best[0]:
        best = (ll, lr, model)
_, best_lr_standard, std_model = best
print(f"-> standard best lr = {best_lr_standard}")
torch.save(std_model.state_dict(), "models/standard_torch.pt")

std_val = predict_standard(std_model, Xt_val)
std_test = predict_standard(std_model, Xt_test)

# ------------------------------------------------------ compare and save
models = {
    "baseline": (base_val, base_test),
    "sklearn": (sk_val, sk_test),
    "manual_torch": (man_val, man_test),
    "standard_torch": (std_val, std_test),
}

results = {}
for name, (pv, pt) in models.items():
    thr = best_threshold(y_val, pv)        # threshold chosen on validation only
    results[name] = evaluate(y_test, pt, thr)  # reported on test

table = pd.DataFrame(results).T[
    ["roc_auc", "pr_auc", "threshold", "cost", "cost_per_loan", "tn", "fp", "fn", "tp"]
]
print("\nTEST RESULTS")
print(table.round(4).to_string())

pd.DataFrame({
    "y_true": y_test,
    **{name: pt for name, (_, pt) in models.items()},
}).to_csv("models/test_predictions.csv", index=False)

json.dump(
    {
        "results": results,
        "config": {
            "fn_cost": FN_COST, "fp_cost": FP_COST, "best_C": best_C,
            "lam": lam, "lr_manual": best_lr_manual, "lr_standard": best_lr_standard,
            "manual_epochs": MANUAL_EPOCHS, "standard_epochs": STANDARD_EPOCHS,
            "seed": SEED, "base_rate": base_rate,
        },
    },
    open("models/results.json", "w"),
    indent=2,
)

train=10499, val=2251, test=2250, features=28
sklearn C=0.001: val log loss 0.3205
sklearn C=0.01: val log loss 0.3094
sklearn C=0.1: val log loss 0.3085
sklearn C=1.0: val log loss 0.3085
sklearn C=10.0: val log loss 0.3085
-> sklearn best C = 0.1
manual lr=0.01: val log loss 0.3250
manual lr=0.1: val log loss 0.3098
manual lr=1.0: val log loss 0.3085
-> manual best lr = 1.0
standard lr=0.001: val log loss 0.3197
standard lr=0.01: val log loss 0.3097
standard lr=0.1: val log loss 0.3165
-> standard best lr = 0.01

TEST RESULTS
                roc_auc  pr_auc  threshold    cost  cost_per_loan      tn      fp    fn     tp
baseline         0.5000  0.1160       0.01  1989.0         0.8840     0.0  1989.0   0.0  261.0
sklearn          0.7560  0.3164       0.10  1378.0         0.6124  1241.0   748.0  63.0  198.0
manual_torch     0.7557  0.3168       0.10  1378.0         0.6124  1241.0   748.0  63.0  198.0
standard_torch   0.7485  0.3096       0.09  1387.0         0.6164  1132.0   857.0  53.